# Energy Forecast Studio: uji mundur di Kaggle

Notebook ini menjalankan pipeline yang sama dengan repositori `muhshandy-dev/energy-forecast-studio`:

1. Unduh data Open Power System Data (time series 2020-10-06, CC BY 4.0) dan cocokkan SHA-256 dengan `data/manifest.json`.
2. Seleksi konfigurasi di validasi 2018, evaluasi uji 2019 sampai September 2020 sekali, lalu prakiraan perencanaan.
3. Bandingkan `reports/results.json` hasil Kaggle dengan versi di repositori.

Pengaturan Kaggle: **Internet: On**, akselerator tidak perlu (CPU cukup, sekitar 40 menit).

In [ ]:
!git clone --depth 1 https://github.com/muhshandy-dev/energy-forecast-studio.git
%cd energy-forecast-studio
!pip install -q -e ".[dev]"

In [ ]:
!energy-forecast data --source opsd

In [ ]:
!cp reports/results.json /kaggle/working/results_repo.json
!energy-forecast run --source opsd

In [ ]:
import json
a = json.load(open("/kaggle/working/results_repo.json"))
b = json.load(open("reports/results.json"))
same = a["metrics"] == b["metrics"]
print("Metrik sama persis dengan repositori:", same)
for name, m in b["metrics"]["test"].items():
    if name == "n_origins":
        continue
    w = m["week_ahead"]
    print(f"{name:10s} MAE7={w['mae']:8.0f}  MASE={w['mase']:.3f}  cov80={m['interval']['80']['coverage']:.3f}  cov95={m['interval']['95']['coverage']:.3f}")

In [ ]:
!python -m pytest -q
!energy-forecast export-web && python tools/verify_published.py